# 07 — Conclusions, limitations, data dictionary (Phase 8)

## Research question
How did a film's Bechdel test rating and its dialogue-by-gender breakdown
relate to box office success over 1990–2015? (Association, not causation.)

## Conclusions

1. **Passing the Bechdel test is associated with ~22% higher domestic gross**
   at fixed budget and year (+21.7%, 95% CI +6.3% to +39.3%, p = .004,
   N = 798). The association is stable across worldwide gross (+20.9%),
   inflation-adjusted gross (+18.5%), and genre controls (+17.4%).
2. **The association is invisible without the budget control.** The raw
   pass/fail gross gap is ≈ 0 because passing films are made on ~25% smaller
   budgets (mean $52M vs $69M for rating-0 films) and budget strongly drives
   gross (r = .66). Holding budget fixed reveals that passing films earn more
   than their budgets predict — a suppression pattern. This vindicates the
   three-dataset design: without The Numbers' budgets, the headline finding
   would have been "no relationship."
3. **Threshold beats quantity.** Female dialogue *share* shows no meaningful
   association with gross (+0.7% per +10pp, CI −3.2% to +4.7%). Whatever the
   Bechdel test captures — women interacting with each other about something
   other than a man — it is not reducible to the amount of female speech.
4. Business-facing phrasing (still associational): over 1990–2015 there is no
   evidence that films centering women underperformed; if anything they
   over-delivered relative to the budgets studios gave them.

## Limitations (first-class, not footnotes)

1. **Triple selection funnel.** The sample is the intersection of three
   coverage filters, each favoring big commercial films: Pudding's script
   collection (1,587 window films, mean infl-adj gross $93M) → Bechdel
   volunteer coverage (1,189, $106M) → budget-data coverage (798, $125M).
   Findings describe box-office variation WITHIN already-successful films;
   they say nothing about films that were never widely released. Notably, the
   films lost at each merge were MORE female-forward (e.g., unmatched-to-
   Numbers films: 33.6% female dialogue vs 28.3% matched), so the sample
   under-represents exactly the small female-forward films the question is
   about.
2. **Association, not causation.** No design here supports causal claims;
   unobserved confounders (star power, marketing spend, franchise status)
   are uncontrolled.
3. **Binary gender coding.** The Pudding data codes characters m/f only;
   5 uncodable characters were dropped.
4. **Bechdel source vintage.** Ratings come from a March-2021 mirror of
   bechdeltest.com (the live API's bulk endpoint was down throughout the
   project — confirmed on a clean network). Films rated after 2021 are
   missing and the `dubious` flag was unavailable.
5. **2015 cutoff.** The Pudding corpus ends in 2015; nothing here speaks to
   post-2015 shifts in representation or box-office dynamics.
6. **Nominal-dollar outcome.** The primary outcome is nominal gross with year
   as a control (absorbing price drift); the inflation-adjusted robustness
   check (R2, +18.5%) confirms the finding is not an inflation artifact.
7. **Fuzzy-join residual risk.** 15 of 798 matches (1.9%) came from fuzzy
   matching; all were score ≥ 92 or individually hand-verified, and the
   cross-source gross correlation (r = .981) bounds the plausible
   contamination at a very low level.

## Provenance chain (full detail: `data/raw/README_provenance.md`)

| Source | Vintage | Role |
|---|---|---|
| Pudding film-dialogue corpus (matthewfdaniels/scripts) | 2016 | dialogue metrics, film spine |
| bechdeltest.com via TidyTuesday mirror | Mar 2021 | Bechdel ratings |
| The Numbers via TidyTuesday mirror | Oct 2018 | budgets/grosses ≤ $175M |
| The Numbers via WWU DATA 311 mirror | early 2023 | budgets/grosses, top-1000 |

## Data dictionary — `data/interim/analysis_table.csv` (798 rows)

The code cell below builds the dictionary and ASSERTS it matches the actual
table column-for-column, so it can never silently drift.

In [1]:
from pathlib import Path

import pandas as pd

df = pd.read_csv(Path.cwd().parent / "data" / "interim" / "analysis_table.csv")

D = [  # column, source, description
    ("imdb_id", "Pudding meta_data7", "IMDb id, tt+7 digits; primary key; join key for Bechdel"),
    ("script_id", "Pudding", "Pudding's internal film id (bridges character_list5 to meta_data7)"),
    ("title", "Pudding meta_data7", "film title as recorded by Pudding"),
    ("year", "Pudding meta_data7", "release year per Pudding; window filter 1990-2015 applied"),
    ("gross_infl_adj_musd", "Pudding meta_data7", "domestic gross, inflation-adjusted $M (2016 basis); robustness outcome only"),
    ("total_words", "derived: character_list5", "sum of words over speaking characters (gender m/f, words>0)"),
    ("f_words", "derived: character_list5", "sum of words spoken by female-coded characters"),
    ("n_characters", "derived: character_list5", "number of speaking characters after exclusions"),
    ("n_female", "derived: character_list5", "number of female-coded speaking characters"),
    ("pct_female_dialogue", "derived", "f_words / total_words (0-1); M2 predictor"),
    ("pct_female_characters", "derived", "n_female / n_characters (0-1)"),
    ("female_lead", "derived", "True if the character with the most words is female-coded"),
    ("bechdel_rating", "Bechdel mirror 2021", "0-3 ordinal: 2+ women / who talk / about something besides a man"),
    ("bechdel_year", "Bechdel mirror 2021", "year per bechdeltest.com (validation: 99.9% within +/-1 of Pudding year)"),
    ("bechdel_pass", "derived", "bechdel_rating == 3; M1 primary predictor"),
    ("production_budget", "Numbers union", "production budget, nominal USD; central control (log10 in models)"),
    ("domestic_gross", "Numbers union", "US/Canada gross, nominal USD; PRIMARY outcome (log10)"),
    ("worldwide_gross", "Numbers union", "worldwide gross, nominal USD; robustness outcome R1"),
    ("distributor", "Numbers union (TT rows only)", "distributor; missing for WWU-sourced rows"),
    ("mpaa_rating", "Numbers union (TT rows only)", "MPAA rating; missing for WWU-sourced rows"),
    ("genre", "Numbers union (TT rows only)", "single primary genre; robustness control R4 (N=711)"),
    ("source", "Numbers union", "which mirror the budget row came from (tidytuesday_2018 / wwu_mirror_2023)"),
    ("match_method", "04 fuzzy join", "audit: exact / year_pm1 / fuzzy — how the Numbers row was matched"),
]
dd = pd.DataFrame(D, columns=["column", "source", "description"])

missing_from_dd = set(df.columns) - set(dd["column"])
extra_in_dd = set(dd["column"]) - set(df.columns)
assert not missing_from_dd, f"table columns not documented: {missing_from_dd}"
assert not extra_in_dd, f"documented columns not in table: {extra_in_dd}"
print(f"data dictionary matches analysis_table.csv: {len(dd)} columns documented\n")

dd.to_csv(Path.cwd().parent / "data" / "interim" / "data_dictionary.csv", index=False)
pd.set_option("display.max_colwidth", 100)
dd

data dictionary matches analysis_table.csv: 23 columns documented



,column,source,description
0,imdb_id,Pudding meta_data7,"IMDb id, tt+7 digits; primary key; join key for Bechdel"
1,script_id,Pudding,Pudding's internal film id (bridges character_list5 to meta_data7)
2,title,Pudding meta_data7,film title as recorded by Pudding
3,year,Pudding meta_data7,release year per Pudding; window filter 1990-2015 applied
4,gross_infl_adj_musd,Pudding meta_data7,"domestic gross, inflation-adjusted $M (2016 basis); robustness outcome only"
5,total_words,derived: character_list5,"sum of words over speaking characters (gender m/f, words>0)"
6,f_words,derived: character_list5,sum of words spoken by female-coded characters
7,n_characters,derived: character_list5,number of speaking characters after exclusions
8,n_female,derived: character_list5,number of female-coded speaking characters
9,pct_female_dialogue,derived,f_words / total_words (0-1); M2 predictor


## Attribution

- Bechdel ratings: bechdeltest.com (CC BY-NC 3.0), via the TidyTuesday
  2021-03-09 mirror.
- Film dialogue: Hannah Anderson & Matt Daniels, *The Pudding* — "Film
  Dialogue from 2,000 screenplays, Broken Down by Gender and Age" (2016),
  data via github.com/matthewfdaniels/scripts.
- Budgets & grosses: The Numbers (the-numbers.com), via the TidyTuesday
  2018-10-23 mirror and the WWU DATA 311 (W23) course mirror.